## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

## 0. Zones, keys and settlement / sector classes (shared by this notebook's tests)

* **Super-zone** of a TAZ: the keys table the chain uses (`Input/Matrices/1270_..._keys.csv`, decision of 4 October 2026).
* **Locality type** (guideline 3.2: urban / suburban / rural): `SETTL_SIZE` of `Zonal_2020.csv`, 3 = urban, 2 = suburban, 1 = rural. This reading is a working assumption from the densities of the three classes
  (population per area unit 2,715 / 319 / 54); the field has no dictionary in the repository.
* **Sector of a household / person** (guideline 3.2): `HHTYPE` of the household table: Arab = `arab`, Haredi = `orto`, Jewish non-Haredi = `secular` + `religious` + `other` (decision D2). The matrices carry no sector, so where a matrix is split by sector
  the TAZ is assigned the sector with the largest share of its population (`SECTOR_ARA`, `SECTOR_ORT`, `SECTOR_REST` of the zonal file): a stated proxy.

In [2]:
def afile(name): return f'{OUT}/{name}.csv'                                   # period-independent outputs (T1, T2)
KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(KEYS, 'Input/TAZ_2636_Keys.xlsx', 'Input/Submatrix_tazs.xlsx', 'Input/Corridor_TAZ_Agg_V2.xlsx', 'Input/TAZ_GSnew.csv')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID'); zon25 = pd.read_csv('Input/Zonal_BU_2025.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
taz_sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW'].reindex(TAZ).astype(int)
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ); Z1250 = sorted(taz_1250.unique())
d_ = pd.DataFrame({'z': taz_1250.values, 'g': taz_sz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values}, index=TAZ)
sz_of_1250 = pd.Series({z: int(g.groupby('g')['pop'].sum().idxmax()) for z, g in d_.groupby('z')})
SZL = sorted(taz_sz.unique()); sz_names = pd.read_csv('Input/sz_localities.csv').set_index('SZ_NEW')['main localities']
LOCNAME = {1: 'rural', 2: 'suburban', 3: 'urban'}
taz_loc = zon.loc[TAZ, 'SETTL_SIZE'].map(LOCNAME)
SECTORS = ['Jewish non-Haredi', 'Haredi', 'Arab']
sec_share = zon.loc[TAZ, ['SECTOR_REST', 'SECTOR_ORT', 'SECTOR_ARA']].fillna(0); sec_share.columns = SECTORS
taz_sector = sec_share.idxmax(axis=1)                                                # proxy: dominant sector of the TAZ population
sub = pd.read_excel('Input/Submatrix_tazs.xlsx'); taz_area = sub.set_index('TAZ')['AggAreaCode']
corr_taz = set(sub.loc[sub['IsLRT_Corridor'] == 1, 'TAZ']) & set(TAZ)
v2 = pd.read_excel('Input/Corridor_TAZ_Agg_V2.xlsx', sheet_name='TazAgg').set_index('TAZ')['AggCode']; gs = pd.read_csv('Input/TAZ_GSnew.csv').set_index('TAZ')['GS']
SECTOR_OF_HHTYPE = {'arab': 'Arab', 'orto': 'Haredi'}                                # everything else: Jewish non-Haredi
print(f'{len(TAZ)} study TAZs, {len(SZL)} super-zones, {len(Z1250)} 1250-zones; locality types {taz_loc.value_counts().to_dict()}; dominant sector {taz_sector.value_counts().to_dict()}')

778 study TAZs, 36 super-zones, 396 1250-zones; locality types {'urban': 400, 'suburban': 219, 'rural': 159}; dominant sector {'Jewish non-Haredi': 568, 'Arab': 200, 'Haredi': 10}


# Ministry of Transport validation, wave 2b: mode split, car occupancy, Metronit boardings, convergence (T9, T10, T16, T19)

**Question.** Does the calibration and growth (RavKav anchoring of bus, Furness to grown margins) keep the survey's mode split and occupancy, does the survey's Metronit travel have the order of magnitude and the spatial pattern of the
2025 smart-card boardings, and did the balancing converge? As in wave 2a these are mostly consistency checks of steps that were added to the survey; they say whether those steps preserved it, not that a model reproduces independent data.
**Not run:** link-level Metronit loads (the 2025 files hold boardings only, no alightings), mode-choice model parameters (no model is estimated: demand is the survey layer), all comparisons with CBS.
Frozen version: 23 September 2026 rebuild; AM 06:00-09:00, PM 16:00-19:00, midday 10:00-14:00.

In [3]:
two, thr = 'Output/ths2017/two_mode' + SUF, 'Output/ths2017/three_mode_2022' + SUF
def sq(p): m = pd.read_csv(p, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
def sqt(p): return sq(p).reindex(index=TAZ, columns=TAZ).fillna(0)
need(f'{two}/car_taz.csv', f'{two}/bus_survey_taz.csv', f'{two}/bus_calibrated_taz.csv', f'{two}/taxi_survey_taz.csv', f'{two}/rail_survey_taz.csv', *[f'{thr}/{n}_2022_taz.csv' for n in ['car', 'bus', 'taxi', 'rail']])
MODES = ['car', 'bus', 'taxi', 'rail']
L_sv = dict(zip(MODES, [sqt(f'{two}/car_taz.csv'), sqt(f'{two}/bus_survey_taz.csv'), sqt(f'{two}/taxi_survey_taz.csv'), sqt(f'{two}/rail_survey_taz.csv')]))
L_cal = {**L_sv, 'bus': sqt(f'{two}/bus_calibrated_taz.csv')}
L_22 = {m: sqt(f'{thr}/{m}_2022_taz.csv') for m in MODES}
def seg_shares(L, key, order, side):
    g = np.asarray(key); out = {}
    for k in order:
        idx = np.where(g == k)[0]
        out[k] = pd.Series({m: (L[m].values[idx, :].sum() if side == 'o' else L[m].values[:, idx].sum()) for m in MODES})
    return pd.DataFrame(out).T
print({n: {m: round(float(L[m].values.sum())) for m in MODES} for n, L in [('survey 2018', L_sv), ('calibrated 2018', L_cal), ('final 2022', L_22)]})

{'survey 2018': {'car': 1198961, 'bus': 81398, 'taxi': 5512, 'rail': 4540}, 'calibrated 2018': {'car': 1198961, 'bus': 87354, 'taxi': 5512, 'rail': 4540}, 'final 2022': {'car': 1259603, 'bus': 88817, 'taxi': 5790, 'rail': 3599}}


## 1. T9 - mode split by segment (guideline 3.6)

**Test.** Mode shares of the motorised trips (car, bus, taxi-type, rail) by origin super-zone, destination super-zone, locality type and sector proxy: the raw survey layers (reference) against the RavKav-calibrated 2018 layers and the final 2022 layers. With four modes of
which one carries ~90 %, the coincidence ratio is lenient; the transit share in percentage points is shown with it and is the more demanding number. **Criterion:** CR above 0.6 for each segment (the plan's consistency criterion; the guideline gives no mode-split limit in this draft), and the transit-share change in pp is reported, not tested.
The mode choice itself is not modelled: the chain keeps the surveyed mode of each trip, so a match here shows that calibration and growth did not distort it, nothing more.

In [4]:
def seg_table(key, order, side, name):
    s0, s1, s2 = (seg_shares(L, key, order, side) for L in (L_sv, L_cal, L_22)); rows = []
    for k in order:
        if s0.loc[k].sum() <= 0: continue
        t = lambda s: 1 - s.loc[k, 'car'] / s.loc[k].sum()
        rows.append({'segment': k, 'trips (survey)': s0.loc[k].sum(), 'trips (2022)': s2.loc[k].sum(), 'transit share survey': t(s0), 'transit share calibrated 2018': t(s1), 'transit share 2022': t(s2),
                     'CR survey vs calibrated': vm.coincidence_ratio(s0.loc[k], s1.loc[k]), 'CR survey vs 2022': vm.coincidence_ratio(s0.loc[k], s2.loc[k])})
    d = pd.DataFrame(rows); d['transit share 2022 - survey (pp)'] = 100 * (d['transit share 2022'] - d['transit share survey']); d['CR above 0.6'] = (d[['CR survey vs calibrated', 'CR survey vs 2022']].min(axis=1) > 0.6)
    d.to_csv(vfile(f'T9_modesplit_{name}'), index=False, float_format='%.4f'); return d
t9o = seg_table(taz_sz.values, SZL, 'o', 'by_origin_sz'); t9d = seg_table(taz_sz.values, SZL, 'd', 'by_destination_sz')
t9l = seg_table(taz_loc.values, ['urban', 'suburban', 'rural'], 'o', 'by_locality'); t9s = seg_table(taz_sector.values, SECTORS, 'o', 'by_sector')
tot = [pd.Series({m: L[m].values.sum() for m in MODES}) for L in (L_sv, L_cal, L_22)]
print(f'{PLABEL} study area: transit share survey {1 - tot[0].car / tot[0].sum():.1%}, calibrated 2018 {1 - tot[1].car / tot[1].sum():.1%}, 2022 {1 - tot[2].car / tot[2].sum():.1%}; CR survey vs 2022 {vm.coincidence_ratio(tot[0], tot[2]):.3f}')
for n, d in [('origin SZ', t9o), ('destination SZ', t9d), ('locality', t9l), ('sector', t9s)]:
    print(f'{n}: segments with CR above 0.6 in both comparisons {int(d["CR above 0.6"].sum())} of {len(d)}; min CR {d[["CR survey vs calibrated", "CR survey vs 2022"]].min().min():.3f}; largest transit-share change {d["transit share 2022 - survey (pp)"].abs().max():.1f} pp')
print(t9l.round(3).to_string(index=False)); print(t9s.round(3).to_string(index=False))
print(t9o.reindex(t9o['transit share 2022 - survey (pp)'].abs().sort_values(ascending=False).index).head(6)[['segment', 'trips (survey)', 'transit share survey', 'transit share calibrated 2018', 'transit share 2022', 'CR survey vs 2022']].round(3).to_string(index=False))

PM 16:00-19:00 study area: transit share survey 7.1%, calibrated 2018 7.5%, 2022 7.2%; CR survey vs 2022 0.995
origin SZ: segments with CR above 0.6 in both comparisons 36 of 36; min CR 0.808; largest transit-share change 9.8 pp
destination SZ: segments with CR above 0.6 in both comparisons 36 of 36; min CR 0.880; largest transit-share change 6.3 pp
locality: segments with CR above 0.6 in both comparisons 3 of 3; min CR 0.958; largest transit-share change 1.8 pp
sector: segments with CR above 0.6 in both comparisons 3 of 3; min CR 0.942; largest transit-share change 2.3 pp
 segment  trips (survey)  trips (2022)  transit share survey  transit share calibrated 2018  transit share 2022  CR survey vs calibrated  CR survey vs 2022  transit share 2022 - survey (pp)  CR above 0.6
   urban      720456.735    762115.488                 0.079                          0.083               0.080                    0.992              0.997                             0.114          True
suburban    

In [5]:
cr_all = vm.coincidence_ratio(tot[0], tot[2])
add_summary('T9', '3.6', f'mode split of motorised trips, study area, 2022 vs survey ({PLABEL})', 'study area', 'CR above 0.6', f'CR {cr_all:.3f}; transit share {1 - tot[0].car / tot[0].sum():.1%} (survey) -> {1 - tot[1].car / tot[1].sum():.1%} (RavKav-calibrated 2018) -> {1 - tot[2].car / tot[2].sum():.1%} (2022)',
            vm.verdict(bool(cr_all > 0.6), explained=True), 'The transit share moves by design: bus is calibrated to RavKav and rail to the national series.', 'matrices')
for n, d, lv in [('origin super-zone', t9o, 'super-zones'), ('destination super-zone', t9d, 'super-zones'), ('locality type', t9l, 'locality types'), ('sector proxy', t9s, 'sectors')]:
    bad = d[~d['CR above 0.6']]
    add_summary('T9', '3.6', f'mode-split CR by {n} ({PLABEL})', f'{len(d)} {lv}', 'each CR above 0.6', f'{int(d["CR above 0.6"].sum())} of {len(d)} above 0.6; minimum {d[["CR survey vs calibrated", "CR survey vs 2022"]].min().min():.2f}; largest transit-share change {d["transit share 2022 - survey (pp)"].abs().max():.1f} pp',
                vm.verdict(bool(d['CR above 0.6'].all()), explained=True), ('Below 0.6: ' + ', '.join(str(x) for x in bad['segment']) + '. ' if len(bad) else '') + 'Where the CR falls, it is where the RavKav anchoring raises or lowers the bus volume or the growth differs by mode.', 'matrices')
add_summary('T9', '3.6', 'generalized-cost weights: out-of-vehicle to in-vehicle time ratio', 'GC function', 'ratio 2 to 3', 'access walk 2, wait 2, in-vehicle 1 (ratio 2); transfer penalty 8 minutes', 'documented',
            'Inside the guideline range at its lower end. The weights were not estimated here; the cost-sensitivity lambda was estimated at person level in step 31 (METHODOLOGY 6ae).', 'METHODOLOGY 6x, 6ae')
add_summary('T9', '3.6', 'mode-choice model parameters (nesting, constants, elasticities) and calibration targets', 'model', 'plausible signs and magnitudes', 'not applicable: no mode-choice model is estimated; the surveyed mode of each trip is kept', 'not applicable', 'The LRT capture step applies a logit with a fixed cost sensitivity to the survey layer; its parameters are tested in the capture sensitivity tables, not here.', '')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), facecolor='white')
scatter_origin(axes[0], t9o['transit share survey'], t9o['transit share 2022'], 'Transit share by origin super-zone', 'survey layers', 'final 2022 layers', color=BLUE)
scatter_origin(axes[1], t9d['transit share survey'], t9d['transit share 2022'], 'Transit share by destination super-zone', 'survey layers', 'final 2022 layers', color=AQUA); save_fig(fig, 'mot_T9_modesplit.png')

## 2. T10 - car occupancy (guideline 3.7)

**Test.** Occupancy = person trips by car (driver + passenger) per driver trip, from the expanded survey (`new_wf`), for the window, by origin super-zone, locality type and sector proxy, against the single value the chain uses to turn person trips into vehicles
(the diagnostics convert the person-trip layers to vehicles for the count comparison of T12; the matrices themselves stay in person trips). **Criterion:** each segment within ±10 % of the study-area value, and the vehicle-trip pattern by origin super-zone unchanged (CR above 0.95) when segment occupancy replaces the constant.
The survey's own reported number of accompanying persons (`numAccomp`) gives a second estimate.
**Correction found by this test (4 October 2026).** The AM occupancy of 1.33 used in step 36 (and in the first version of T12, and the PM and midday values 1.43 and 1.32 of wave 1b) took the hour from the `Dep_h` of the row of the *arrival* place, which is the hour the person leaves the destination. The departure hour of a trip is the `Dep_h` of the *previous* place of the person-day
(arrival time minus the previous departure time equals the reported travel time), the rule steps 15 and 16 use for the matrices. With the right hour the AM occupancy is 1.52. The matrices were never affected; T12 is rerun with the window's own occupancy and the step-36 notebook keeps its original number as a record.

In [6]:
df = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx'); k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636'); map_2636_1250 = k26.set_index('TAZ_2636')['TAZ_1250']
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy(); g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1); d['trip_dep_h'] = g['Dep_h'].shift(1)
tw = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h']); tw = tw[tw['trip_dep_h'].isin(HOURS)].copy()
tw['o1250'] = tw['origin'].map(map_2636_1250); tw['d1250'] = tw['actTaz'].map(map_2636_1250); tw['osz'] = tw['o1250'].map(sz_of_1250)
tw['oloc'] = tw['o1250'].map(pd.Series(taz_loc.values, index=taz_1250.values).groupby(level=0).agg(lambda s: s.value_counts().index[0]))
tw['osec'] = tw['o1250'].map(pd.Series(taz_sector.values, index=taz_1250.values).groupby(level=0).agg(lambda s: s.value_counts().index[0]))
car = tw[tw['mainmode'].isin([10, 11]) & tw['o1250'].isin(Z1250) & tw['d1250'].isin(Z1250)].copy()
def occ(x): return x.loc[x.mainmode.isin([10, 11]), 'new_wf'].sum() / x.loc[x.mainmode == 10, 'new_wf'].sum() if x.loc[x.mainmode == 10, 'new_wf'].sum() > 0 else np.nan
def occ2(x): dr = x[x.mainmode == 10]; return 1 + np.average(dr['numAccomp'].fillna(0), weights=dr['new_wf']) if dr['new_wf'].sum() > 0 else np.nan
OCC_ALL = occ(car); OCC_ALL2 = occ2(car)
_o = df[df.Dep_h.isin(HOURS)]; OCC_OLD = float(_o[_o.mainmode.isin([10, 11])]['new_wf'].sum() / _o[_o.mainmode == 10]['new_wf'].sum()); OCC_WIN = occ(tw)       # OCC_OLD: the arrival-row hour (step 36 / wave 1b); OCC_WIN: all window trips, right hour (used by the corrected T12)
rows = []
for lab, col, order in [('origin super-zone', 'osz', SZL), ('locality type', 'oloc', ['urban', 'suburban', 'rural']), ('sector proxy', 'osec', SECTORS)]:
    for k in order:
        x = car[car[col] == k]
        if (x.mainmode == 10).sum() < 30: continue
        rows.append({'dimension': lab, 'segment': k, 'sampled driver trips': int((x.mainmode == 10).sum()), 'occupancy (drivers + passengers)': occ(x), 'occupancy (1 + numAccomp)': occ2(x)})
t10 = pd.DataFrame(rows); t10['diff to study area %'] = 100 * (t10['occupancy (drivers + passengers)'] / OCC_ALL - 1); t10['within ±10 %'] = t10['diff to study area %'].abs() <= 10
t10.to_csv(vfile('T10_occupancy'), index=False, float_format='%.4f')
print(f'{PLABEL}: occupancy {OCC_ALL:.3f} (trips inside the study area; {int((car.mainmode == 10).sum()):,} sampled driver trips; all window trips {OCC_WIN:.3f}), 1 + numAccomp {OCC_ALL2:.3f}; value used before the correction (arrival-row hour) {OCC_OLD:.3f}')
print(t10[t10.dimension != 'origin super-zone'].round(3).to_string(index=False)); sz_ = t10[t10.dimension == 'origin super-zone']; print(f'super-zones with at least 30 sampled driver trips: {len(sz_)}; within ±10 % of the study area: {int(sz_["within ±10 %"].sum())}')
# vehicle-trip pattern by origin super-zone: constant vs segment occupancy
car_o = pd.Series({k: L_sv['car'].values[np.where(taz_sz.values == k)[0], :].sum() for k in SZL}); oc_sz = sz_.set_index('segment')['occupancy (drivers + passengers)'].reindex(SZL).fillna(OCC_ALL)
v_const, v_seg = car_o / OCC_ALL, car_o / oc_sz
cr_occ = vm.coincidence_ratio(v_const, v_seg); print(f'vehicle-trip pattern by origin super-zone, constant vs segment occupancy: CR {cr_occ:.4f}; vehicle total {v_const.sum():,.0f} vs {v_seg.sum():,.0f} ({100 * (v_seg.sum() / v_const.sum() - 1):+.1f} %)')

PM 16:00-19:00: occupancy 1.525 (trips inside the study area; 9,607 sampled driver trips; all window trips 1.518), 1 + numAccomp 1.445; value used before the correction (arrival-row hour) 1.430
    dimension           segment  sampled driver trips  occupancy (drivers + passengers)  occupancy (1 + numAccomp)  diff to study area %  within ±10 %
locality type             urban                  5098                             1.572                      1.492                 3.059          True
locality type          suburban                  2971                             1.544                      1.460                 1.195          True
locality type             rural                  1538                             1.304                      1.231               -14.535         False
 sector proxy Jewish non-Haredi                  7235                             1.458                      1.390                -4.392          True
 sector proxy            Haredi                   1

In [7]:
add_summary('T10', '3.7', f'car occupancy of the window ({PLABEL}), study area, against the value the diagnostics used before this test', 'study area', 'within ±10 %', f'survey, right departure hour: {OCC_ALL:.3f} (all window trips {OCC_WIN:.3f}; 1 + numAccomp {OCC_ALL2:.3f}); value used before: {OCC_OLD:.3f}',
            vm.verdict(bool(abs(OCC_WIN / OCC_OLD - 1) <= 0.10), explained=True), 'Explained: the earlier value took the departure hour of the destination place (see the correction note above). T12 is rerun with the window value; the car-to-count ratio falls accordingly.', 'THS 2017/18')
for dim in ['origin super-zone', 'locality type', 'sector proxy']:
    x = t10[t10.dimension == dim]
    add_summary('T10', '3.7', f'occupancy by {dim} ({PLABEL})', f'{len(x)} segments', 'each within ±10 % of the study-area value', f'{int(x["within ±10 %"].sum())} of {len(x)} within ±10 %; range {x["occupancy (drivers + passengers)"].min():.2f}-{x["occupancy (drivers + passengers)"].max():.2f}',
                vm.verdict(bool(x['within ±10 %'].all()), explained=True), 'Where it differs, the sampled driver trips are few (at least 30 required) and the occupancy of shared cars differs between sectors; a segment occupancy would move the vehicle totals of that segment.' if not x['within ±10 %'].all() else '', 'THS 2017/18')
add_summary('T10', '3.7', 'vehicle-trip pattern by origin super-zone, constant vs segment occupancy', f'{len(SZL)} super-zones', 'CR above 0.95', f'CR {cr_occ:.4f}; vehicle total {100 * (v_seg.sum() / v_const.sum() - 1):+.1f} % with segment occupancy', vm.verdict(bool(cr_occ > 0.95), explained=True), '', 'THS 2017/18')
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor='white'); x = sz_.sort_values('occupancy (drivers + passengers)')
ax.bar(range(len(x)), x['occupancy (drivers + passengers)'], color=BLUE); ax.axhline(OCC_ALL, color=ORANGE, lw=1.2); ax.set_xticks(range(len(x))); ax.set_xticklabels(x['segment'].astype(int), fontsize=7); ax.set_ylim(1, None)
style_ax(ax, f'Car occupancy by origin super-zone, {PLABEL} (orange: study area)', 'super-zone', 'persons per car'); fig.tight_layout(); save_fig(fig, 'mot_T10_occupancy.png')

## 3. T16 - Metronit boardings (guideline 5.3, station-level analogue)

**Test.** The survey's Metronit travel (main mode 5, the AM window, expanded) against the 2025 smart-card boardings of the Haifa BRT (`Output/ravkav_2025/boardings_by_taz_2025.csv`: Tuesday 06:00-08:59, averaged over 51-52 Tuesdays, transfer and non-transfer boardings separate), by super-zone.
The survey frame is the *origin of the person trip* (doorstep), the smart-card frame the *boarding stop*, and a person trip can have more than one boarding (transfers); the survey is 2017/18 and the cards 2025, after a COVID dip and the Metronit network growth. The comparison therefore says whether the survey layer has the level and spatial pattern of the BRT demand, not that it reproduces it.
**Criteria:** the guideline's station-demand classes (RMSE% by size) as the analogue for super-zone sums; the total within ±15 %. Only the AM window has 2025 cards (PM and midday: not run).

In [8]:
if PERIOD != 'AM':
    print('2025 smart-card boardings are available for the AM window only: T16 is not run for', PLABEL)
else:
    rk = pd.read_csv('Output/ravkav_2025/boardings_by_taz_2025.csv').set_index('TAZ').reindex(TAZ).fillna(0)
    met = tw[(tw['mainmode'] == 5) & tw['o1250'].isin(Z1250)].copy(); met['osz'] = met['osz'].astype(int)
    sv_sz = met.groupby('osz')['new_wf'].sum().reindex(SZL).fillna(0)
    rk_all = rk['metronit_all'].groupby(taz_sz.values).sum().reindex(SZL); rk_nt = rk['metronit_nontransfer'].groupby(taz_sz.values).sum().reindex(SZL)
    t16 = pd.DataFrame({'survey Metronit trips (origin zone)': sv_sz, 'sampled trips': met.groupby('osz').size().reindex(SZL).fillna(0).astype(int), 'RavKav 2025 boardings, non-transfer': rk_nt, 'RavKav 2025 boardings, all': rk_all}); t16.index.name = 'super-zone'
    t16 = t16[(t16.iloc[:, 0] > 0) | (t16.iloc[:, 2] > 0)]; t16.to_csv(vfile('T16_metronit_by_superzone'), float_format='%.2f')
    fa = vm.fit_stats(t16['RavKav 2025 boardings, non-transfer'], t16['survey Metronit trips (origin zone)']); fb = vm.fit_stats(t16['RavKav 2025 boardings, all'], t16['survey Metronit trips (origin zone)'])
    cls = vm.rmse_by_class(t16['RavKav 2025 boardings, non-transfer'].values, t16['survey Metronit trips (origin zone)'].values, vm.STATION_RMSE_CLASSES)
    print(f'survey Metronit trips (study area origins) {t16.iloc[:, 0].sum():,.0f} ({int(t16["sampled trips"].sum())} sampled) vs 2025 boardings non-transfer {t16.iloc[:, 2].sum():,.0f}, all {t16.iloc[:, 3].sum():,.0f}')
    print(f'vs non-transfer: ratio {t16.iloc[:, 0].sum() / t16.iloc[:, 2].sum():.2f}, R2 {fa["R2"]:.3f}, slope {fa["slope_origin"]:.2f}, RMSE% {fa["RMSE_pct"]:.0f}; vs all boardings: ratio {t16.iloc[:, 0].sum() / t16.iloc[:, 3].sum():.2f}, R2 {fb["R2"]:.3f}, RMSE% {fb["RMSE_pct"]:.0f}')
    print(pd.DataFrame(cls).round(1).to_string(index=False)); print(t16.sort_values('RavKav 2025 boardings, non-transfer', ascending=False).head(10).round(0).to_string())
    cr16 = vm.coincidence_ratio(t16['RavKav 2025 boardings, non-transfer'], t16['survey Metronit trips (origin zone)']); print(f'CR of the super-zone distributions: {cr16:.3f}')

2025 smart-card boardings are available for the AM window only: T16 is not run for PM 16:00-19:00


In [9]:
if PERIOD == 'AM':
    cr16_ = cr16
    add_summary('T16', '5.3', 'Metronit demand, level: survey AM trips (origin zone) vs 2025 smart-card non-transfer boardings, study area', 'study area', 'within ±15 %', f'survey {t16.iloc[:, 0].sum():,.0f} vs cards {t16.iloc[:, 2].sum():,.0f} ({100 * (t16.iloc[:, 0].sum() / t16.iloc[:, 2].sum() - 1):+.0f} %); against all boardings including transfers {t16.iloc[:, 3].sum():,.0f} ({100 * (t16.iloc[:, 0].sum() / t16.iloc[:, 3].sum() - 1):+.0f} %)',
                vm.verdict(bool(abs(t16.iloc[:, 0].sum() / t16.iloc[:, 2].sum() - 1) <= 0.15), explained=True), 'Different years (2017/18 vs 2025, network growth after the survey), different frames (doorstep origin vs boarding stop, sampled trips few).', 'THS 2017/18; RavKav 2025')
    add_summary('T16', '5.3', 'Metronit demand, spatial pattern by super-zone', f'{len(t16)} super-zones', 'RMSE% in the station-demand class limits; CR reported', f'R2 {fa["R2"]:.2f}, slope {fa["slope_origin"]:.2f}, RMSE% {fa["RMSE_pct"]:.0f}; CR {cr16_:.2f}; classes: ' + '; '.join(f'{c["class"]}: {c["RMSE_pct"]:.0f} % (limit {c["limit_pct"]:.0f}, n {c["n"]})' for c in cls),
                vm.verdict(all(c['pass'] for c in cls), explained=True), 'A person trip by Metronit starts at the doorstep, the card at the stop; access by bus or on foot moves the boarding to another super-zone, and the survey sample for the mode is small.', 'THS 2017/18; RavKav 2025')
    add_summary('T16', '5.3', 'Metronit link loads / boardings by station against counts', 'stations / links', 'guideline 5.3 classes', 'not run: the 2025 files hold boardings only (no alightings), and station counts of the operator are not in the repository', 'not run', 'Operator 22 (Metronit) data request, open question 5.', '')
    fig, ax = plt.subplots(figsize=(6.2, 4.8), facecolor='white'); scatter_origin(ax, t16['RavKav 2025 boardings, non-transfer'], t16['survey Metronit trips (origin zone)'], 'Metronit AM by super-zone', 'RavKav 2025 non-transfer boardings (observed)', 'survey Metronit trips (model)', color=PURPLE); save_fig(fig, 'mot_T16_metronit.png')
else:
    add_summary('T16', '5.3', f'Metronit boardings ({PLABEL})', 'super-zone', 'guideline 5.3 classes', 'not run: the 2025 smart-card extract used here covers 06:00-08:59 only', 'not run', 'Extract the PM and midday hours of the 2025 files to run it.', '')

## 4. T19 - convergence of the balancing (guideline 3.8)

**Test.** The 2018 -> 2022 car and taxi-type layers are Furnessed to grown margins (origins x population growth, destinations x employment growth, grand total set by the origin side). The test recomputes the balancing from the saved 2018 layers and growth factors with the same code as step 16,
and checks (a) that it reproduces the saved 2022 layers, (b) the iterations to converge, (c) the largest residual of the row and column margins against their targets, and (d) that the aggregations to super-zones and areas carry the same volumes as the TAZ matrices. **Criterion:** margins within 1 % of their targets in every zone with a target of at least 100 trips (a stated convergence tolerance; the guideline does not give one for matrix balancing),
and the saved layers reproduced to the rounding of the files.

In [10]:
gf = pd.read_csv(f'{thr}/growth_factors_taz_2018_2022.csv').set_index('TAZ').reindex(TAZ); g_o, g_d = gf['g_origin'].values, gf['g_dest'].values
def furness(seed, row_t, col_t, iters=500, tol=1e-6):
    m = np.asarray(seed, float).copy()
    for it in range(iters):
        rs = m.sum(axis=1); m *= np.divide(row_t, rs, out=np.zeros_like(row_t), where=rs > 0)[:, None]
        cs = m.sum(axis=0); m *= np.divide(col_t, cs, out=np.ones_like(col_t), where=cs > 0)[None, :]
        if max(np.abs(m.sum(axis=1) - row_t).max(), np.abs(m.sum(axis=0) - col_t).max()) < tol: break
    return m, it + 1
def grow(seed):
    row_t = seed.sum(axis=1) * g_o; col_t = seed.sum(axis=0) * g_d; col_t = col_t * row_t.sum() / col_t.sum()
    out, it = furness(seed, row_t, col_t); return out, it, row_t, col_t
rows = []
for name, seed, saved in [('car', L_sv['car'], L_22['car']), ('taxi-type', L_sv['taxi'], L_22['taxi'])]:
    out, it, rt, ct = grow(seed.values); rs, cs = out.sum(axis=1), out.sum(axis=0)
    rbig, cbig = rt >= 100, ct >= 100
    rel_r = np.abs(rs[rbig] - rt[rbig]) / rt[rbig]; rel_c = np.abs(cs[cbig] - ct[cbig]) / ct[cbig]
    short = rt - rs
    rows.append({'layer': name, 'iterations': it, 'trips 2018': seed.values.sum(), 'trips 2022 (recomputed)': out.sum(), 'trips 2022 (saved)': saved.values.sum(), 'max abs difference to saved layer': np.abs(out - saved.values).max(),
                 'max row residual (trips)': np.abs(rs - rt).max(), 'max column residual (trips)': np.abs(cs - ct).max(), 'rows with target >= 100': int(rbig.sum()), 'max relative row residual, targets >= 100': rel_r.max() if rbig.any() else np.nan,
                 'max relative column residual, targets >= 100': rel_c.max() if cbig.any() else np.nan, 'row shortfall (trips, total)': short[short > 0].sum(), 'share of layer': short[short > 0].sum() / rt.sum()})
t19 = pd.DataFrame(rows); t19.to_csv(vfile('T19_furness_convergence'), index=False, float_format='%.6g'); print(t19.T.to_string())
chk = []
for lay, tz_ in [('car', L_22['car']), ('bus', L_22['bus']), ('taxi', L_22['taxi']), ('rail', L_22['rail'])]:
    for lvl, fn, m_ in [('super-zone', f'{lay}_2022_sz.csv', taz_sz.values)]:
        p = f'{thr}/{fn}'
        if not os.path.exists(p): continue
        s_ = sq(p); a = pd.DataFrame(tz_.values, index=m_, columns=m_).groupby(level=0).sum().T.groupby(level=0).sum().T
        a = a.reindex(index=s_.index, columns=s_.columns).fillna(0); chk.append({'layer': lay, 'level': lvl, 'TAZ total': tz_.values.sum(), 'aggregate total': s_.values.sum(), 'max abs cell difference': np.abs(a.values - s_.values).max()})
agg = pd.DataFrame(chk); agg.to_csv(vfile('T19_aggregation_check'), index=False, float_format='%.6g'); print(agg.to_string(index=False))

                                                           0            1
layer                                                    car    taxi-type
iterations                                               256          500
trips 2018                                    1198961.373064  5511.798634
trips 2022 (recomputed)                       1259602.652337  5790.357543
trips 2022 (saved)                            1259602.503977  5790.321704
max abs difference to saved layer                   0.129403     0.020773
max row residual (trips)                            0.000001     93.01738
max column residual (trips)                              0.0          0.0
rows with target >= 100                                  664           16
max relative row residual, targets >= 100                0.0     0.271607
max relative column residual, targets >= 100             0.0          0.0
row shortfall (trips, total)                        0.000017   278.650242
share of layer                        

In [11]:
for _, r in t19.iterrows():
    ok = bool(r['max relative row residual, targets >= 100'] <= 0.01 and r['max relative column residual, targets >= 100'] <= 0.01 and r['max abs difference to saved layer'] < 0.5)
    add_summary('T19', '3.8', f'Furness balancing of the {r["layer"]} layer, 2018 to 2022 ({PLABEL})', 'TAZ matrix', 'margins within 1 % of target (zones with target >= 100 trips); saved layer reproduced to the file rounding (0.5 trips)',
                f'{int(r["iterations"])} iterations; largest relative residual rows {r["max relative row residual, targets >= 100"]:.2%}, columns {r["max relative column residual, targets >= 100"]:.2%}; recomputed vs saved layer: largest cell difference {r["max abs difference to saved layer"]:.2g} trips (files hold 6 significant figures); row shortfall {r["row shortfall (trips, total)"]:,.0f} trips ({r["share of layer"]:.2%} of the layer)',
                vm.verdict(ok, explained=True), 'Cells of zones whose growth target is zero cannot be scaled up: the row shortfall is the volume of such rows; the saved layers are exactly the recomputed ones.' if not ok else '', 'step 16 code, saved layers')
add_summary('T19', '3.8', 'aggregation TAZ -> super-zone carries the same volumes', 'four layers', 'identical totals and cells to the file rounding (0.5 trips)', '; '.join(f'{r["layer"]}: total difference {r["TAZ total"] - r["aggregate total"]:.2g}, largest cell {r["max abs cell difference"]:.2g}' for _, r in agg.iterrows()),
            vm.verdict(bool((agg['max abs cell difference'] < 0.5).all()), explained=True), '', 'saved layers')

## Summary of this notebook against the guideline

In [12]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage2c{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T9,3.6,"mode split of motorised trips, study area, 202...",study area,CR above 0.6,CR 0.995; transit share 7.1% (survey) -> 7.5% ...,pass
1,T9,3.6,mode-split CR by origin super-zone (PM 16:00-1...,36 super-zones,each CR above 0.6,36 of 36 above 0.6; minimum 0.81; largest tran...,pass
2,T9,3.6,mode-split CR by destination super-zone (PM 16...,36 super-zones,each CR above 0.6,36 of 36 above 0.6; minimum 0.88; largest tran...,pass
3,T9,3.6,mode-split CR by locality type (PM 16:00-19:00),3 locality types,each CR above 0.6,3 of 3 above 0.6; minimum 0.96; largest transi...,pass
4,T9,3.6,mode-split CR by sector proxy (PM 16:00-19:00),3 sectors,each CR above 0.6,3 of 3 above 0.6; minimum 0.94; largest transi...,pass
5,T9,3.6,generalized-cost weights: out-of-vehicle to in...,GC function,ratio 2 to 3,"access walk 2, wait 2, in-vehicle 1 (ratio 2);...",documented
6,T9,3.6,"mode-choice model parameters (nesting, constan...",model,plausible signs and magnitudes,not applicable: no mode-choice model is estima...,not applicable
7,T10,3.7,"car occupancy of the window (PM 16:00-19:00), ...",study area,within ±10 %,"survey, right departure hour: 1.525 (all windo...",pass
8,T10,3.7,occupancy by origin super-zone (PM 16:00-19:00),36 segments,each within ±10 % of the study-area value,22 of 36 within ±10 %; range 1.28-2.08,miss (explained)
9,T10,3.7,occupancy by locality type (PM 16:00-19:00),3 segments,each within ±10 % of the study-area value,2 of 3 within ±10 %; range 1.30-1.57,miss (explained)
